## Notebook 07 — Baseline LightGBM Model

---

This notebook:
1. Loads + merges data, drops >80% missing columns, removes `TransactionID`
2. Performs stratified 80/20 train-validation split
3. Converts object columns to 'category' dtype (for LightGBM compatibility)
4. Trains baseline LightGBM classifier with early stopping
5. Evaluates on validation set (ROC-AUC, PR-AUC)
6. Displays best iteration and top 20 feature importances

> **No imputation or encoding is performed. Only dtype conversion for categorical features.**

---
## 📦 Step 0 — Import Libraries, Load & Split Data

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../Datasets/'
TARGET    = 'isFraud'

# Load & merge
print('Loading and merging...')
train_transaction = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(DATA_PATH + 'train_identity.csv')
train_raw = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_transaction, train_identity

# Drop >80% missing columns
missing_pct  = train_raw.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()
train = train_raw.drop(columns=cols_to_drop)
del train_raw

# Drop identifier column
train = train.drop(columns=['TransactionID'], errors='ignore')

# Stratified 80/20 split
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
del train, X, y

print(f'✅ X_train shape : {X_train.shape}')
print(f'✅ X_val shape   : {X_val.shape}')
print(f'✅ Train fraud % : {y_train.mean()*100:.2f}%')
print(f'✅ Val fraud %   : {y_val.mean()*100:.2f}%')

Loading and merging...
✅ X_train shape : (472432, 358)
✅ X_val shape   : (118108, 358)
✅ Train fraud % : 3.50%
✅ Val fraud %   : 3.50%


---
## 🔄 Step 1 — Convert Object Columns to Category Dtype

In [17]:
# Identify object columns
object_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f'Found {len(object_cols)} object columns')
print(f'Converting to category dtype...')
print()

# Convert to category dtype in both train and validation sets
for col in object_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

print(f'✅ Converted {len(object_cols)} categorical columns in X_train and X_val')
print()
print('Categorical columns:')
for i, col in enumerate(object_cols, 1):
    print(f'  {i:2d}. {col}')

Found 26 object columns
Converting to category dtype...

✅ Converted 26 categorical columns in X_train and X_val

Categorical columns:
   1. ProductCD
   2. card4
   3. card6
   4. P_emaildomain
   5. R_emaildomain
   6. M1
   7. M2
   8. M3
   9. M4
  10. M5
  11. M6
  12. M7
  13. M8
  14. M9
  15. id_12
  16. id_15
  17. id_16
  18. id_28
  19. id_29
  20. id_31
  21. id_35
  22. id_36
  23. id_37
  24. id_38
  25. DeviceType
  26. DeviceInfo


---
## 🚀 Step 2 — Initialize LightGBM Model

In [18]:
# Initialize LightGBM classifier with specified parameters
model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    scale_pos_weight=27.6,
    random_state=42,
    verbose=-1
)

print('✅ LightGBM model initialized')
print(f'   Objective: binary')
print(f'   Metric: auc')
print(f'   Max estimators: 1000')
print(f'   Learning rate: 0.05')
print(f'   Num leaves: 64')
print(f'   Scale pos weight: 27.6')
print(f'   Early stopping rounds: 100')

✅ LightGBM model initialized
   Objective: binary
   Metric: auc
   Max estimators: 1000
   Learning rate: 0.05
   Num leaves: 64
   Scale pos weight: 27.6
   Early stopping rounds: 100


---
## 🎯 Step 3 — Train Model with Early Stopping

In [19]:
print('Training LightGBM model...')
print()

# Train with early stopping
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

print()
print('✅ Training complete')

Training LightGBM model...

Training until validation scores don't improve for 100 rounds
[50]	valid_0's auc: 0.92496
[100]	valid_0's auc: 0.938345
[150]	valid_0's auc: 0.945205
[200]	valid_0's auc: 0.949704
[250]	valid_0's auc: 0.952877
[300]	valid_0's auc: 0.955505
[350]	valid_0's auc: 0.957204
[400]	valid_0's auc: 0.959421
[450]	valid_0's auc: 0.960672
[500]	valid_0's auc: 0.962008
[550]	valid_0's auc: 0.963193
[600]	valid_0's auc: 0.96417
[650]	valid_0's auc: 0.964984
[700]	valid_0's auc: 0.965722
[750]	valid_0's auc: 0.966259
[800]	valid_0's auc: 0.966865
[850]	valid_0's auc: 0.96736
[900]	valid_0's auc: 0.967859
[950]	valid_0's auc: 0.968224
[1000]	valid_0's auc: 0.968701
Did not meet early stopping. Best iteration is:
[997]	valid_0's auc: 0.968709

✅ Training complete


---
## 📊 Step 4 — Evaluate on Validation Set

In [20]:
# Get predictions on validation set
y_val_pred_proba = model.predict_proba(X_val)[:, 1]

# Compute metrics
roc_auc = roc_auc_score(y_val, y_val_pred_proba)
pr_auc = average_precision_score(y_val, y_val_pred_proba)

# Get best iteration
best_iter = model.best_iteration_

# Print results
print('=' * 60)
print('   VALIDATION METRICS')
print('=' * 60)
print(f'Validation ROC-AUC:        {roc_auc:.6f}')
print(f'Validation PR-AUC:         {pr_auc:.6f}')
print(f'Best Iteration:            {best_iter}')
print('=' * 60)

   VALIDATION METRICS
Validation ROC-AUC:        0.968709
Validation PR-AUC:         0.822145
Best Iteration:            997


---
## 🏆 Step 5 — Top 20 Feature Importances

In [21]:
# Get feature importances
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

feature_importance.index += 1
top20 = feature_importance.head(20)

# Print top 20 features
W = 70
sep = '=' * W
dash = '-' * W
hdr = f'{"Rank":<6} {"Feature":<30} {"Importance":>20}'

print(sep)
print('   TOP 20 FEATURES — By Importance (Gain)')
print(sep)
print(hdr)
print(dash)
for rank, row in top20.iterrows():
    print(
        f'{rank:<6} {row["Feature"]:<30} '
        f'{row["Importance"]:>20,.2f}'
    )
print(sep)

   TOP 20 FEATURES — By Importance (Gain)
Rank   Feature                                  Importance
----------------------------------------------------------------------
1      card1                                      3,708.00
2      TransactionDT                              3,701.00
3      card2                                      3,462.00
4      addr1                                      3,457.00
5      TransactionAmt                             3,087.00
6      P_emaildomain                              2,499.00
7      id_31                                      2,324.00
8      D15                                        1,577.00
9      dist1                                      1,499.00
10     DeviceInfo                                 1,487.00
11     C13                                        1,385.00
12     D4                                         1,320.00
13     card5                                      1,244.00
14     D10                                        1,207.00
15

---
## 🎯 Step 6 — Threshold Analysis

In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Define thresholds to evaluate
thresholds = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

# Store results
results = []

for threshold in thresholds:
    # Apply threshold
    y_val_pred = (y_val_pred_proba >= threshold).astype(int)
    
    # Compute metrics
    precision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(y_val, y_val_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Fraud metrics (same as precision/recall for positive class)
    fraud_precision = precision
    fraud_recall = recall  # This is TPR (True Positive Rate)
    
    results.append({
        'Threshold': threshold,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Fraud Recall (TPR)': fraud_recall,
        'Fraud Precision': fraud_precision,
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp
    })

# Print results table
W = 110
sep = '=' * W
dash = '-' * W
hdr = (f'{"Threshold":<11} {"Precision":>11} {"Recall":>11} {"F1-Score":>11} '
       f'{"Fraud Recall":>13} {"Fraud Prec":>12} {"TN":>8} {"FP":>8} {"FN":>8} {"TP":>8}')

print(sep)
print('   THRESHOLD ANALYSIS — Validation Set Performance')
print(sep)
print(hdr)
print(dash)

for res in results:
    print(
        f'{res["Threshold"]:<11.2f} '
        f'{res["Precision"]:>11.4f} '
        f'{res["Recall"]:>11.4f} '
        f'{res["F1-Score"]:>11.4f} '
        f'{res["Fraud Recall (TPR)"]:>13.4f} '
        f'{res["Fraud Precision"]:>12.4f} '
        f'{res["TN"]:>8,} '
        f'{res["FP"]:>8,} '
        f'{res["FN"]:>8,} '
        f'{res["TP"]:>8,}'
    )
print(sep)

# Print default threshold (0.5) metrics separately
print()
print('=' * 60)
print('   DEFAULT THRESHOLD (0.5) METRICS')
print('=' * 60)
default_result = [r for r in results if r['Threshold'] == 0.5][0]
print(f'Precision:              {default_result["Precision"]:.6f}')
print(f'Recall:                 {default_result["Recall"]:.6f}')
print(f'F1-Score:               {default_result["F1-Score"]:.6f}')
print(f'Fraud Recall (TPR):     {default_result["Fraud Recall (TPR)"]:.6f}')
print(f'Fraud Precision:        {default_result["Fraud Precision"]:.6f}')
print()
print('Confusion Matrix:')
print(f'  TN: {default_result["TN"]:>8,}  |  FP: {default_result["FP"]:>8,}')
print(f'  FN: {default_result["FN"]:>8,}  |  TP: {default_result["TP"]:>8,}')
print('=' * 60)

   THRESHOLD ANALYSIS — Validation Set Performance
Threshold     Precision      Recall    F1-Score  Fraud Recall   Fraud Prec       TN       FP       FN       TP
--------------------------------------------------------------------------------------------------------------
0.05             0.0846      0.9777      0.1558        0.9777       0.0846   70,262   43,713       92    4,041
0.10             0.1271      0.9625      0.2245        0.9625       0.1271   86,649   27,326      155    3,978
0.20             0.2156      0.9289      0.3500        0.9289       0.2156  100,012   13,963      294    3,839
0.30             0.3112      0.8972      0.4621        0.8972       0.3112  105,767    8,208      425    3,708
0.40             0.4137      0.8614      0.5589        0.8614       0.4137  108,929    5,046      573    3,560
0.50             0.5261      0.8326      0.6447        0.8326       0.5261  110,875    3,100      692    3,441

   DEFAULT THRESHOLD (0.5) METRICS
Precision:              0

---
## 🔁 Step 7 — Stratified 5-Fold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Prepare full training data (combine X_train and X_val back)
# We need to reload and prepare the data for CV
print('Preparing data for cross-validation...')

# Reload data
train_transaction = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(DATA_PATH + 'train_identity.csv')
train_raw = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_transaction, train_identity

# Drop >80% missing columns
missing_pct  = train_raw.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()
train = train_raw.drop(columns=cols_to_drop)
del train_raw

# Drop identifier column
train = train.drop(columns=['TransactionID'], errors='ignore')

# Separate features and target
X_full = train.drop(columns=[TARGET])
y_full = train[TARGET]
del train

# Convert object columns to category dtype
object_cols_cv = X_full.select_dtypes(include=['object']).columns.tolist()
for col in object_cols_cv:
    X_full[col] = X_full[col].astype('category')

print(f'✅ Data prepared: {X_full.shape}')
print(f'✅ Target fraud rate: {y_full.mean()*100:.2f}%')
print()

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
cv_results = []

print('Starting 5-Fold Cross Validation...')
print('=' * 80)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_full, y_full), 1):
    print(f'\nFold {fold_idx}/5')
    print('-' * 80)
    
    # Split data
    X_train_fold = X_full.iloc[train_idx]
    y_train_fold = y_full.iloc[train_idx]
    X_val_fold = X_full.iloc[val_idx]
    y_val_fold = y_full.iloc[val_idx]
    
    print(f'  Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}')
    print(f'  Train fraud: {y_train_fold.mean()*100:.2f}% | Val fraud: {y_val_fold.mean()*100:.2f}%')
    
    # Initialize model for this fold
    fold_model = lgb.LGBMClassifier(
        objective='binary',
        metric='auc',
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=64,
        scale_pos_weight=27.6,
        random_state=42,
        verbose=-1
    )
    
    # Train with early stopping
    fold_model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False)
        ]
    )
    
    # Predict on validation fold
    y_val_pred_proba_fold = fold_model.predict_proba(X_val_fold)[:, 1]
    
    # Compute metrics
    fold_roc_auc = roc_auc_score(y_val_fold, y_val_pred_proba_fold)
    fold_pr_auc = average_precision_score(y_val_fold, y_val_pred_proba_fold)
    fold_best_iter = fold_model.best_iteration_
    
    print(f'  Best iteration: {fold_best_iter}')
    print(f'  ROC-AUC: {fold_roc_auc:.6f}')
    print(f'  PR-AUC:  {fold_pr_auc:.6f}')
    
    # Store results
    cv_results.append({
        'Fold': fold_idx,
        'ROC-AUC': fold_roc_auc,
        'PR-AUC': fold_pr_auc,
        'Best Iteration': fold_best_iter
    })

print()
print('=' * 80)
print('   CROSS-VALIDATION RESULTS SUMMARY')
print('=' * 80)
print()

# Print fold-by-fold results
print('Fold-by-Fold Results:')
print(f'{"Fold":<6} {"ROC-AUC":>12} {"PR-AUC":>12} {"Best Iter":>12}')
print('-' * 45)
for res in cv_results:
    print(f'{res["Fold"]:<6} {res["ROC-AUC"]:>12.6f} {res["PR-AUC"]:>12.6f} {res["Best Iteration"]:>12}')
print()

# Compute statistics
roc_aucs = [res['ROC-AUC'] for res in cv_results]
pr_aucs = [res['PR-AUC'] for res in cv_results]

mean_roc_auc = np.mean(roc_aucs)
std_roc_auc = np.std(roc_aucs)
mean_pr_auc = np.mean(pr_aucs)
std_pr_auc = np.std(pr_aucs)

# Print summary statistics
print('Summary Statistics:')
print('-' * 45)
print(f'Mean ROC-AUC:      {mean_roc_auc:.6f} ± {std_roc_auc:.6f}')
print(f'Mean PR-AUC:       {mean_pr_auc:.6f} ± {std_pr_auc:.6f}')
print()
print(f'ROC-AUC Range:     [{min(roc_aucs):.6f}, {max(roc_aucs):.6f}]')
print(f'PR-AUC Range:      [{min(pr_aucs):.6f}, {max(pr_aucs):.6f}]')
print('=' * 80)